In [15]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("="*80)
print("DATA PREPROCESSING PIPELINE")
print("="*80)

DATA PREPROCESSING PIPELINE


In [16]:
# Cell 2: Load Data
df = pd.read_csv('MCI_Challenge_FinalDataset.csv')
print(f"Initial dataset shape: {df.shape}")
print(f"Number of columns: {len(df.columns)}")

Initial dataset shape: (7043, 17)
Number of columns: 17


In [17]:
# Cell 3: Remove Unnecessary Columns & Handle Missing Values
print("="*80)
print("STEP 1: INITIAL CLEANING")
print("="*80)

# Remove customer ID
if 'شناسه_مشترک' in df.columns:
    df = df.drop('شناسه_مشترک', axis=1)
    print("Customer ID removed")

# Check missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values found")
else:
    print(f"Missing values: {missing[missing>0]}")
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if df[col].dtype in ['int64', 'float64']:
                df[col].fillna(df[col].median(), inplace=True)
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
    print("Missing values filled")

# Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"{duplicates} duplicate records removed")

print(f"\nShape after cleaning: {df.shape}")

STEP 1: INITIAL CLEANING
Customer ID removed
No missing values found

Shape after cleaning: (7043, 16)


In [18]:
# Cell 4: Transform Target Variable (Churn)
print("="*80)
print("STEP 2: TARGET VARIABLE TRANSFORMATION")
print("="*80)

# Map churn to 0/1
df['ریزش'] = df['ریزش'].map({'خیر': 0, 'بله': 1})
churn_rate = df['ریزش'].mean() * 100
print(f"Churn rate in entire dataset: {churn_rate:.1f}%")
print(f"   No churn: {(df['ریزش']==0).sum():,} customers")
print(f"   Churn: {(df['ریزش']==1).sum():,} customers")

STEP 2: TARGET VARIABLE TRANSFORMATION
Churn rate in entire dataset: 26.5%
   No churn: 5,174 customers
   Churn: 1,869 customers


In [19]:
# Cell 5: Demographic Features
print("="*80)
print("STEP 3: DEMOGRAPHIC FEATURES")
print("="*80)

# Gender - Label Encoding
df['جنسیت'] = df['جنسیت'].map({'زن': 0, 'مرد': 1})
print("Gender: 0=Female, 1=Male")

# Age - numeric
df['سن'] = df['سن'].astype(float)
print("Age: numeric")



STEP 3: DEMOGRAPHIC FEATURES
Gender: 0=Female, 1=Male
Age: numeric


In [20]:
# Cell 6: Birth Month Cyclical Encoding
print("="*80)
print("STEP 4: BIRTH MONTH CYCLICAL ENCODING")
print("="*80)

# Month mapping
month_to_num = {
    'فروردین': 1, 'اردیبهشت': 2, 'خرداد': 3,
    'تیر': 4, 'مرداد': 5, 'شهریور': 6,
    'مهر': 7, 'آبان': 8, 'آذر': 9,
    'دی': 10, 'بهمن': 11, 'اسفند': 12
}

# Convert month to number
df['ماه_عدد'] = df['ماه_تولد'].map(month_to_num)

# Convert to sine and cosine
angle = 2 * np.pi * (df['ماه_عدد'] - 1) / 12
df['ماه_سینوسی'] = np.sin(angle)
df['ماه_کسینوسی'] = np.cos(angle)

# Drop original columns
df = df.drop(['ماه_تولد', 'ماه_عدد'], axis=1)

print("Birth month: transformed to sine/cosine features")
print(f"   - Farvardin (month 1): sin(0)={np.sin(0):.2f}, cos(0)={np.cos(0):.2f}")
print(f"   - Tir (month 4): sin(90 deg)=1, cos(90 deg)=0")
print(f"   - Mehr (month 7): sin(180 deg)=0, cos(180 deg)=-1")
print(f"   - Dey (month 10): sin(270 deg)=-1, cos(270 deg)=0")

STEP 4: BIRTH MONTH CYCLICAL ENCODING
Birth month: transformed to sine/cosine features
   - Farvardin (month 1): sin(0)=0.00, cos(0)=1.00
   - Tir (month 4): sin(90 deg)=1, cos(90 deg)=0
   - Mehr (month 7): sin(180 deg)=0, cos(180 deg)=-1
   - Dey (month 10): sin(270 deg)=-1, cos(270 deg)=0


In [21]:
# Cell 7: Internet Generation - One-Hot Encoding
print("="*80)
print("STEP 5: INTERNET GENERATION (ONE-HOT ENCODING)")
print("="*80)

unique_gens = df['نسل_اینترنت_همراه'].unique()
print(f"Unique internet generation values: {unique_gens}")

# One-Hot Encoding
df = pd.get_dummies(df, columns=['نسل_اینترنت_همراه'], prefix='gen', drop_first=False)

print("Internet generation: One-Hot Encoding applied")

STEP 5: INTERNET GENERATION (ONE-HOT ENCODING)
Unique internet generation values: ['4G' '5G' '2G' '3G']
Internet generation: One-Hot Encoding applied


In [22]:
# Cell 8: Service Columns Transformation (No: -1, Yes: 1, N/A: 0)
print("="*80)
print("STEP 6: SERVICE COLUMNS TRANSFORMATION")
print("="*80)

# List ALL your service columns explicitly
service_cols = [
    'بسته_رومینگ_بین‌الملل',
    'فضای_ابری_اپراتور', 
    'بسته_اینترنت_شبانه',
    'سرویس_تماس_VoLTE',
    'سوپراپ_شبکه اجتماعی',
    'سوپراپ_خدمات_مالی'
]

print(f"Found {len(service_cols)} service columns:")
for col in service_cols:
    print(f"  - {col}")

# Transform: No -> -1, Yes -> 1, others (N/A, etc.) -> 0
for col in service_cols:
    if col in df.columns:
        if df[col].dtype == 'object':
            # Create mapping dictionary
            mapping = {'خیر': -1, 'بله': 1}
            # For any other values (like 'ندارد', 'نامشخص', etc.), set to 0
            df[col] = df[col].map(mapping).fillna(0)
            # Convert to int for consistency
            df[col] = df[col].astype(int)
            print(f"   {col}: No=-1, Yes=1, Other=0")
        else:
            print(f"   {col}: already numeric, skipping")
    else:
        print(f"   WARNING: {col} not found in dataframe!")

# SIM card type
if 'نوع_سیم‌کارت' in df.columns:
    df['نوع_سیم‌کارت'] = df['نوع_سیم‌کارت'].map({'اعتباری': 0, 'دائمی': 1})
    print("SIM card type: Prepaid=0, Postpaid=1")

# Verify the transformation
print("\n" + "="*80)
print("VERIFICATION - Sample of transformed service columns:")
print("="*80)
print(df[service_cols].head(10))

STEP 6: SERVICE COLUMNS TRANSFORMATION
Found 6 service columns:
  - بسته_رومینگ_بین‌الملل
  - فضای_ابری_اپراتور
  - بسته_اینترنت_شبانه
  - سرویس_تماس_VoLTE
  - سوپراپ_شبکه اجتماعی
  - سوپراپ_خدمات_مالی
   بسته_رومینگ_بین‌الملل: No=-1, Yes=1, Other=0
   فضای_ابری_اپراتور: No=-1, Yes=1, Other=0
   بسته_اینترنت_شبانه: No=-1, Yes=1, Other=0
   سرویس_تماس_VoLTE: No=-1, Yes=1, Other=0
   سوپراپ_شبکه اجتماعی: No=-1, Yes=1, Other=0
   سوپراپ_خدمات_مالی: No=-1, Yes=1, Other=0
SIM card type: Prepaid=0, Postpaid=1

VERIFICATION - Sample of transformed service columns:
   بسته_رومینگ_بین‌الملل  فضای_ابری_اپراتور  بسته_اینترنت_شبانه  \
0                     -1                  1                  -1   
1                      1                 -1                   1   
2                      1                  1                  -1   
3                      1                 -1                   1   
4                     -1                 -1                  -1   
5                     -1          

In [23]:
# Cell 9: Verify Transformations
print("="*80)
print("STEP 7: VERIFICATION")
print("="*80)

print(f"Final dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")



print(f"\nSample of service columns after transformation:")
if len(service_cols) > 0:
    print(df[service_cols[:5]].head())

STEP 7: VERIFICATION
Final dataset shape: (7043, 20)

Data types:
int64      12
bool        4
float64     3
object      1
Name: count, dtype: int64

Sample of service columns after transformation:
   بسته_رومینگ_بین‌الملل  فضای_ابری_اپراتور  بسته_اینترنت_شبانه  \
0                     -1                  1                  -1   
1                      1                 -1                   1   
2                      1                  1                  -1   
3                      1                 -1                   1   
4                     -1                 -1                  -1   

   سرویس_تماس_VoLTE  سوپراپ_شبکه اجتماعی  
0                -1                   -1  
1                -1                   -1  
2                -1                   -1  
3                 1                   -1  
4                -1                   -1  


In [24]:
df.head

<bound method NDFrame.head of       جنسیت    سن  سابقه_سیم‌کارت_ماه  بسته_رومینگ_بین‌الملل  \
0         0  46.0                   1                     -1   
1         1  38.0                  34                      1   
2         1  48.0                   2                      1   
3         1  58.0                  45                      1   
4         0  37.0                   2                     -1   
...     ...   ...                 ...                    ...   
7038      1  18.0                  24                      1   
7039      0  31.0                  72                     -1   
7040      0  49.0                  11                      1   
7041      1  35.0                   4                     -1   
7042      1  46.0                  66                      1   

      فضای_ابری_اپراتور  بسته_اینترنت_شبانه  سرویس_تماس_VoLTE  \
0                     1                  -1                -1   
1                    -1                   1                -1   
2     

In [25]:
# Cell: Save Processed Dataset
print("="*80)
print("SAVING PROCESSED DATASET")
print("="*80)

# Method 1: Save as CSV (most common)
df.to_csv('MCI_Challenge_Processed.csv', index=False)
print("Saved as: MCI_Challenge_Processed.csv")



SAVING PROCESSED DATASET
Saved as: MCI_Challenge_Processed.csv


In [26]:
# Cell: Convert all Yes/No and True/False to 0/1
print("="*80)
print("CONVERTING ALL YES/NO AND TRUE/FALSE TO 0/1")
print("="*80)

# 1. Convert object type columns that have 'بله'/'خیر'
for col in df.columns:
    if df[col].dtype == 'object':
        if 'بله' in df[col].values or 'خیر' in df[col].values:
            print(f"Converting {col}...")
            df[col] = df[col].map({'خیر': 0, 'بله': 1})
            # Fill any remaining NaN with 0
            df[col] = df[col].fillna(0).astype(int)

# 2. Convert boolean columns (True/False) to 0/1
bool_cols = df.select_dtypes(include=['bool']).columns
for col in bool_cols:
    print(f"Converting boolean {col}...")
    df[col] = df[col].astype(int)

# 3. Check float columns that might contain -1.0, 1.0 (already correct)
# These are fine as they are

print("\n" + "="*80)
print("VERIFICATION AFTER CONVERSION")
print("="*80)

# Show first few rows of converted columns
print("\nSample of converted data:")
print(df.head())

# Show data types after conversion
print("\nData types after conversion:")
print(df.dtypes.value_counts())

# Show which columns are still object type
object_cols = df.select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"\nWARNING: Still have object columns: {object_cols}")
else:
    print("\nSUCCESS: No object columns remaining!")

# Show boolean columns remaining
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"WARNING: Still have boolean columns: {bool_cols}")
else:
    print("SUCCESS: No boolean columns remaining!")

CONVERTING ALL YES/NO AND TRUE/FALSE TO 0/1
Converting استفاده_اپلیکیشن_اپراتور...
Converting boolean gen_2G...
Converting boolean gen_3G...
Converting boolean gen_4G...
Converting boolean gen_5G...

VERIFICATION AFTER CONVERSION

Sample of converted data:
   جنسیت    سن  سابقه_سیم‌کارت_ماه  بسته_رومینگ_بین‌الملل  فضای_ابری_اپراتور  \
0      0  46.0                   1                     -1                  1   
1      1  38.0                  34                      1                 -1   
2      1  48.0                   2                      1                  1   
3      1  58.0                  45                      1                 -1   
4      0  37.0                   2                     -1                 -1   

   بسته_اینترنت_شبانه  سرویس_تماس_VoLTE  سوپراپ_شبکه اجتماعی  \
0                  -1                -1                   -1   
1                   1                -1                   -1   
2                  -1                -1                   -1   
3     

In [27]:
# Cell: Convert ALL column names to English
print("="*80)
print("CONVERTING ALL COLUMN NAMES TO ENGLISH")
print("="*80)

# Show current columns
print("Current columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

# Complete mapping dictionary for all columns
column_mapping = {
    # Demographic
    'جنسیت': 'gender',
    'سن': 'age',
    'سابقه_سیم‌کارت_ماه': 'sim_history_months',
    
    # Birth month
    'ماه_سینوسی': 'birth_month_sin',
    'ماه_کسینوسی': 'birth_month_cos',
    
    # Services
    'بسته_رومینگ_بین‌الملل': 'international_roaming_package',
    'فضای_ابری_اپراتور': 'operator_cloud_space',
    'بسته_اینترنت_شبانه': 'night_internet_package',
    'سرویس_تماس_VoLTE': 'volte_calling_service',
    'سوپراپ_شبکه اجتماعی': 'superapp_social_network',
    'سوپراپ_خدمات_مالی': 'superapp_financial_services',
    'نوع_سیم‌کارت': 'sim_card_type',
    'استفاده_اپلیکیشن_اپراتور': 'operator_app_usage',
    
    # Costs
    'هزینه_ماهیانه_تومان': 'monthly_cost_toman',
    'هزینه_کل_تومان': 'total_cost_toman',
    
    # Target
    'ریزش': 'churn',
    
    # Internet generation (already English, but keeping for safety)
    'gen_2G': 'gen_2G',
    'gen_3G': 'gen_3G',
    'gen_4G': 'gen_4G',
    'gen_5G': 'gen_5G'
}

# Apply renaming
df = df.rename(columns=column_mapping)

print("\n" + "="*80)
print("NEW COLUMN NAMES (ALL ENGLISH):")
print("="*80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

# Verify no Persian characters remain
print("\n" + "="*80)
print("VERIFICATION - Checking for Persian characters:")
print("="*80)

persian_columns = []
for col in df.columns:
    if any('\u0600' <= c <= '\u06FF' for c in col):
        persian_columns.append(col)

if persian_columns:
    print(f"WARNING: Still have Persian columns: {persian_columns}")
else:
    print("SUCCESS: All column names are in English! No Persian characters found.")

CONVERTING ALL COLUMN NAMES TO ENGLISH
Current columns:
 1. جنسیت
 2. سن
 3. سابقه_سیم‌کارت_ماه
 4. بسته_رومینگ_بین‌الملل
 5. فضای_ابری_اپراتور
 6. بسته_اینترنت_شبانه
 7. سرویس_تماس_VoLTE
 8. سوپراپ_شبکه اجتماعی
 9. سوپراپ_خدمات_مالی
10. نوع_سیم‌کارت
11. استفاده_اپلیکیشن_اپراتور
12. هزینه_ماهیانه_تومان
13. هزینه_کل_تومان
14. ریزش
15. ماه_سینوسی
16. ماه_کسینوسی
17. gen_2G
18. gen_3G
19. gen_4G
20. gen_5G

NEW COLUMN NAMES (ALL ENGLISH):
 1. gender
 2. age
 3. sim_history_months
 4. international_roaming_package
 5. operator_cloud_space
 6. night_internet_package
 7. volte_calling_service
 8. superapp_social_network
 9. superapp_financial_services
10. sim_card_type
11. operator_app_usage
12. monthly_cost_toman
13. total_cost_toman
14. churn
15. birth_month_sin
16. birth_month_cos
17. gen_2G
18. gen_3G
19. gen_4G
20. gen_5G

VERIFICATION - Checking for Persian characters:
SUCCESS: All column names are in English! No Persian characters found.


In [ ]:
# Cell: Save as LayaPreprocessing
print("="*80)
print("SAVING AS DataPreprocessing1")
print("="*80)

# Save as CSV
df.to_csv('DataPreprocessing1.csv', index=False, encoding='utf-8-sig')
print("Saved: DataPreprocessing1.csv")


SAVING AS LayaPreprocessing1
Saved: LayaPreprocessing1.csv
